In [1]:
# ── Step 1: Record the numpy version currently loaded in memory ────────────────
# Must happen BEFORE any pip install touches numpy.
import numpy as _np_preload
import subprocess, sys
_np_version = _np_preload.__version__   # e.g. "1.26.4"
print(f"In-memory numpy: {_np_version}")

# ── Step 2: Install all other packages ────────────────────────────────────────
# Some of these (unsloth, ultralytics) may upgrade numpy as a side-effect.
!pip install -q unsloth
!pip install -q triton
!pip install -q "tokenizers>=0.22.0,<=0.23.0"
!pip install -q rank_bm25
!pip install -q deep_translator
!pip install -q ultralytics

# ── Step 3: Force numpy metadata back to the in-memory version ────────────────
# unsloth_zoo does: if importlib.metadata.version("numpy") != sys.modules["numpy"].__version__: raise
# If anything above upgraded numpy, pip's metadata now says 2.x while the kernel
# still holds 1.x in memory → crash. We force-reinstall ONLY numpy (--no-deps)
# so that pip's metadata is updated back to match what is already in memory.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--force-reinstall", "--no-deps", f"numpy=={_np_version}"],
    check=True
)
print(f"🔒 numpy metadata re-pinned to {_np_version} — matches in-memory version.")

In-memory numpy: 1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 832.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

# Using Qwen3VL 8B for finetuning

In [2]:
from unsloth import FastVisionModel
import torch
import os

_INPUT_BASE          = "/kaggle/input/datasets/huyqn12/cropped-zalo"
# English mixed-trained model (video QA + traffic sign QA combined)
_cached_mixed_model  = os.path.join(_INPUT_BASE, "qwen2-vl-7b-mixed-en")

if os.path.isdir(_cached_mixed_model):
    print("✅ Loading cached qwen2-vl-7b-mixed-en (skipping training) ...")
    model, tokenizer = FastVisionModel.from_pretrained(
        _cached_mixed_model,
        load_in_4bit = True,
        use_gradient_checkpointing = "unsloth",
    )
    _skip_training = True
else:
    print("Loading base Qwen3 VL 8B model ...")
    model, tokenizer = FastVisionModel.from_pretrained(
        "/kaggle/input/qwen-3-vl/transformers/8b-instruct/1",
        load_in_4bit = True,
        use_gradient_checkpointing = "unsloth",
    )
    _skip_training = False


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Loading cached qwen2-vl-7b-mixed-en (skipping training) ...
==((====))==  Unsloth 2026.6.7: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

# Preprocess training data

In [3]:
#split
import os
import json
import random
from collections import defaultdict

_INPUT_BASE = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_cached_train_split = os.path.join(_INPUT_BASE, "train_split.json")
_cached_test_split  = os.path.join(_INPUT_BASE, "test_split.json")

if os.path.exists(_cached_train_split) and os.path.exists(_cached_test_split):
    train_split_path = _cached_train_split
    test_split_path  = _cached_test_split
    print(f"\u2705 Using cached splits from input:")
    print(f"- {train_split_path}")
    print(f"- {test_split_path}")
else:
    train_json_path = "/kaggle/input/train3/train/train/train.json"

    with open(train_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_items = data["data"]

    video_dict = defaultdict(list)
    for item in all_items:
        video_dict[item["video_path"]].append(item)

    video_paths = list(video_dict.keys())
    random.seed(42)
    random.shuffle(video_paths)

    split_idx    = int(0.8 * len(video_paths))
    train_videos = video_paths[:split_idx]
    test_videos  = video_paths[split_idx:]

    train_items = []
    for v in train_videos:
        train_items.extend(video_dict[v])

    test_items = []
    for v in test_videos:
        test_items.extend(video_dict[v])

    train_split = {"__count__": len(train_items), "data": train_items}
    test_split  = {"__count__": len(test_items),  "data": test_items}

    train_split_path = "/kaggle/working/train_split.json"
    test_split_path  = "/kaggle/working/test_split.json"

    with open(train_split_path, "w", encoding="utf-8") as f:
        json.dump(train_split, f, indent=2, ensure_ascii=False)

    with open(test_split_path, "w", encoding="utf-8") as f:
        json.dump(test_split, f, indent=2, ensure_ascii=False)

    print(f"\u2705 Done! Split saved:")
    print(f"- {train_split_path} ({len(train_items)} items)")
    print(f"- {test_split_path} ({len(test_items)} items)")


✅ Done! Split saved:
- /kaggle/working/train_split.json (1164 items)
- /kaggle/working/test_split.json (326 items)


In [4]:
# ════════════════════════════════════════════════════════════════════════════════
#  Translation utilities — Vietnamese → English
#  Uses deep_translator with batching + disk caching for Kaggle re-runs.
# ════════════════════════════════════════════════════════════════════════════════
import os
import json
import hashlib
import time
from deep_translator import GoogleTranslator

_INPUT_BASE = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_TRANS_CACHE_INPUT = os.path.join(_INPUT_BASE, "translation_cache.json")
_TRANS_CACHE_PATH = "/kaggle/working/translation_cache.json"

# Load existing cache
if os.path.exists(_TRANS_CACHE_INPUT):
    with open(_TRANS_CACHE_INPUT, "r", encoding="utf-8") as f:
        _trans_cache = json.load(f)
    print(f"✅ Loaded translation cache from input ({len(_trans_cache)} entries)")
elif os.path.exists(_TRANS_CACHE_PATH):
    with open(_TRANS_CACHE_PATH, "r", encoding="utf-8") as f:
        _trans_cache = json.load(f)
    print(f"✅ Loaded translation cache from working ({len(_trans_cache)} entries)")
else:
    _trans_cache = {}
    print("Starting fresh translation cache.")

_translator = GoogleTranslator(source='vi', target='en')

def translate_vi_to_en(text: str) -> str:
    """Translate Vietnamese text to English with caching."""
    if not text or not text.strip():
        return text
    key = hashlib.md5(text.encode()).hexdigest()
    if key in _trans_cache:
        return _trans_cache[key]
    try:
        result = _translator.translate(text)
        _trans_cache[key] = result
        return result
    except Exception as e:
        # On rate limit, wait and retry once
        time.sleep(1)
        try:
            result = _translator.translate(text)
            _trans_cache[key] = result
            return result
        except:
            return text  # fallback to original

def batch_translate(texts: list, batch_size: int = 50) -> list:
    """Translate a list of strings with progress tracking."""
    results = []
    uncached = [(i, t) for i, t in enumerate(texts) if hashlib.md5(t.encode()).hexdigest() not in _trans_cache]
    cached = len(texts) - len(uncached)
    print(f"  {cached}/{len(texts)} already cached. Translating {len(uncached)} new...")
    
    results = [None] * len(texts)
    for i, t in enumerate(texts):
        results[i] = translate_vi_to_en(t)
        if (i + 1) % 100 == 0:
            print(f"  Translated {i+1}/{len(texts)}...")
    
    # Persist cache
    with open(_TRANS_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(_trans_cache, f, ensure_ascii=False)
    return results

def save_translation_cache():
    with open(_TRANS_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(_trans_cache, f, ensure_ascii=False)
    print(f"💾 Saved translation cache ({len(_trans_cache)} entries)")

print("✅ Translation utilities ready.")

✅ Loaded translation cache from input (6223 entries)
✅ Translation utilities ready.


In [5]:
import json
import os

_INPUT_BASE = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_cached_prefixed = os.path.join(_INPUT_BASE, "train_split_prefixed.json")

if os.path.exists(_cached_prefixed):
    train_split_prefixed_path = _cached_prefixed
    print(f"\u2705 Using cached train_split_prefixed.json from input: {train_split_prefixed_path}")
else:
    prefix = "/kaggle/input/train3/train/"

    with open(train_split_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data["data"]:
        item["video_path"] = prefix + item["video_path"]

    train_split_prefixed_path = "/kaggle/working/train_split_prefixed.json"
    with open(train_split_prefixed_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"\u2705 Saved \u2192 {train_split_prefixed_path}")


✅ Saved → /kaggle/working/train_split_prefixed.json


In [6]:
import os
import json
import cv2
import traceback
from tqdm import tqdm

_INPUT_BASE    = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_cached_conv   = os.path.join(_INPUT_BASE, "converted_train_en.json")
_cached_frames = os.path.join(_INPUT_BASE, "train_frames")
_working_conv  = "/kaggle/working/converted_train_en.json"  # generated on previous run

if os.path.exists(_cached_conv):
    frames_save_dir      = _cached_frames if os.path.isdir(_cached_frames) else "/kaggle/working/train_frames"
    converted_train_path = _cached_conv
    print(f"✅ converted_train_en.json found in input — skipping frame extraction + translation.")
    print(f"   frames_save_dir = {frames_save_dir}")
elif os.path.exists(_working_conv):
    frames_save_dir      = "/kaggle/working/train_frames"
    converted_train_path = _working_conv
    print(f"✅ converted_train_en.json found in working — skipping frame extraction + translation.")
else:
    # ---------------------- CONFIG ----------------------
    frames_save_dir = "/kaggle/working/train_frames"
    os.makedirs(frames_save_dir, exist_ok=True)

    def extract_sample_frames(video_path, save_dir, id_prefix, max_frames=5, max_size=2000):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0:
            cap.release()
            return []
        os.makedirs(save_dir, exist_ok=True)
        idxs = [int(i * total / max_frames) for i in range(max_frames)]
        saved_paths = []
        for i, idx in enumerate(idxs):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue
            h, w = frame.shape[:2]
            if max(h, w) > max_size:
                scale = max_size / max(h, w)
                frame = cv2.resize(frame, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
            name = f"{id_prefix}_{i:04d}.jpg"
            path = os.path.join(save_dir, name)
            cv2.imwrite(path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
            saved_paths.append(path)
        cap.release()
        return saved_paths

    # ---------------------- LOAD INPUT JSON ----------------------
    with open(train_split_prefixed_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # ── Translate all unique questions, choices, answers ───────────────────────
    print("Translating training data to English...")
    all_questions = [item["question"] for item in data["data"]]
    all_answers   = [item.get("answer", "") for item in data["data"]]
    all_choices_flat = []
    choice_map = []  # (item_idx, choice_idx)
    for i, item in enumerate(data["data"]):
        for j, c in enumerate(item.get("choices", [])):
            all_choices_flat.append(c)
            choice_map.append((i, j))

    translated_questions = batch_translate(all_questions)
    translated_answers   = batch_translate(all_answers)
    translated_choices   = batch_translate(all_choices_flat)
    save_translation_cache()

    # Rebuild choices per item
    translated_choices_per_item = [[] for _ in data["data"]]
    for flat_idx, (item_idx, _) in enumerate(choice_map):
        translated_choices_per_item[item_idx].append(translated_choices[flat_idx])

    # ---------------------- MAIN LOOP ----------------------
    converted_dataset = []
    for idx, item in enumerate(tqdm(data["data"], desc="Converting training data")):
        video_path = item["video_path"]
        id_prefix  = os.path.splitext(os.path.basename(video_path))[0]
        frame_paths = extract_sample_frames(video_path, frames_save_dir, id_prefix, max_frames=4)

        if not frame_paths:
            continue

        question_en = translated_questions[idx]
        choices_en  = translated_choices_per_item[idx]
        answer_en   = translated_answers[idx]

        choices_text = "\n".join(choices_en)
        user_text = f"{question_en}\n{choices_text}" if choices_en else question_en

        try:
            entry = {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            *[{"type": "image", "image": path} for path in frame_paths],
                            {"type": "text", "text": user_text}
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": answer_en}
                        ],
                    },
                ]
            }
            converted_dataset.append(entry)
        except Exception as e:
            print(f"[error] {id_prefix}: {e}")
            continue

    # ---------------------- SAVE OUTPUT ----------------------
    print(f"Total entries created: {len(converted_dataset)}")
    converted_train_path = _working_conv
    with open(converted_train_path, "w", encoding="utf-8") as f:
        json.dump(converted_dataset, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved English converted dataset to {converted_train_path}")


✅ converted_train_en.json found in input — skipping frame extraction + translation.
   frames_save_dir = /kaggle/input/datasets/huyqn12/cropped-zalo/train_frames


In [7]:
# Load converted English training dataset
import json
import os

_INPUT_BASE    = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_cached_conv   = os.path.join(_INPUT_BASE, "converted_train_en.json")
_cached_frames = os.path.join(_INPUT_BASE, "train_frames")

converted_json_path = _cached_conv if os.path.exists(_cached_conv) else "/kaggle/working/converted_train_en.json"

with open(converted_json_path, "r", encoding="utf-8") as f:
    converted_dataset = json.load(f)

# Patch image paths if loaded from input
if converted_json_path == _cached_conv and os.path.isdir(_cached_frames):
    _old_prefix = "/kaggle/working/train_frames"
    for entry in converted_dataset:
        for msg in entry.get("messages", []):
            for part in msg.get("content", []):
                if part.get("type") == "image" and part["image"].startswith(_old_prefix):
                    part["image"] = part["image"].replace(_old_prefix, _cached_frames)
    print(f"✅ Loaded and patched {len(converted_dataset)} English entries (images → input).")
else:
    print(f"Loaded {len(converted_dataset)} English entries from {converted_json_path}.")

# Split into train/eval for proper training
import random
random.seed(42)
_indices = list(range(len(converted_dataset)))
random.shuffle(_indices)
_split = int(0.9 * len(_indices))
train_dataset = [converted_dataset[i] for i in _indices[:_split]]
eval_dataset  = [converted_dataset[i] for i in _indices[_split:]]
print(f"  Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

✅ Loaded and patched 1164 English entries (images → input).
  Train: 1047 | Eval: 117


# Mixed Training: Video QA + Traffic Sign QA
Both datasets are translated to English, mixed together, and used for a single fine-tuning run.


In [8]:
# Load and translate traffic sign VQA data to English
import json
import os

_INPUT_BASE    = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_cached_vqa_en = os.path.join(_INPUT_BASE, "traffic_vqa_en.jsonl")
_working_vqa_en = "/kaggle/working/traffic_vqa_en.jsonl"  # generated on previous run

if os.path.exists(_cached_vqa_en):
    traffic_vqa_path = _cached_vqa_en
    print(f"✅ Using cached English traffic_vqa from input: {traffic_vqa_path}")
elif os.path.exists(_working_vqa_en):
    traffic_vqa_path = _working_vqa_en
    print(f"✅ Using cached English traffic_vqa from working: {traffic_vqa_path}")
else:
    # First fix image paths
    _cached_vqa = os.path.join(_INPUT_BASE, "traffic_vqa_qwen2vl_fixed.jsonl")
    if os.path.exists(_cached_vqa):
        input_jsonl = _cached_vqa
    else:
        input_jsonl = "/kaggle/input/vqa-sign/traffic_vqa_qwen2vl (1).jsonl"

    old_prefix = "/kaggle/input/traffic_sign_images/"
    new_prefix = "/kaggle/input/traffic-sign-images/traffic_sign_images/"

    # Load all entries
    raw_entries = []
    with open(input_jsonl, "r", encoding="utf-8") as fin:
        for line in fin:
            raw_entries.append(json.loads(line))

    def _normalize_content(msg):
        """Ensure message content is always a list of dicts, never a bare string."""
        c = msg.get("content", [])
        if isinstance(c, str):
            msg["content"] = [{"type": "text", "text": c}]
        elif isinstance(c, list):
            msg["content"] = [
                {"type": "text", "text": item} if isinstance(item, str) else item
                for item in c
            ]

    # Collect all text content that needs translation
    texts_to_translate = []
    text_locations = []  # (entry_idx, msg_idx, content_idx)

    for ei, entry in enumerate(raw_entries):
        for mi, msg in enumerate(entry.get("messages", [])):
            _normalize_content(msg)  # fix bare-string content in-place
            # Fix image paths and collect texts
            if msg.get("role") == "user":
                for ci, content in enumerate(msg.get("content", [])):
                    if content.get("type") == "image":
                        img_path = content["image"]
                        if img_path.startswith(old_prefix):
                            content["image"] = img_path.replace(old_prefix, new_prefix)
                    elif content.get("type") == "text":
                        texts_to_translate.append(content["text"])
                        text_locations.append((ei, mi, ci))
            elif msg.get("role") == "assistant":
                for ci, content in enumerate(msg.get("content", [])):
                    if content.get("type") == "text":
                        texts_to_translate.append(content["text"])
                        text_locations.append((ei, mi, ci))

    print(f"Translating {len(texts_to_translate)} text segments from traffic VQA...")
    translated = batch_translate(texts_to_translate)
    save_translation_cache()

    # Apply translations
    for idx, (ei, mi, ci) in enumerate(text_locations):
        raw_entries[ei]["messages"][mi]["content"][ci]["text"] = translated[idx]

    # Save English version
    traffic_vqa_path = _working_vqa_en
    with open(traffic_vqa_path, "w", encoding="utf-8") as fout:
        for entry in raw_entries:
            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"✅ Saved English traffic VQA to {traffic_vqa_path} ({len(raw_entries)} entries)")


✅ Using cached English traffic_vqa from input: /kaggle/input/datasets/huyqn12/cropped-zalo/traffic_vqa_en.jsonl


In [9]:
import json

sign_data = []
with open(traffic_vqa_path, "r", encoding="utf-8") as f:
    for line in f:
        sign_data.append(json.loads(line))

print(f"Loaded {len(sign_data)} English sign VQA samples.")

# Split for eval
import random
random.seed(42)
_s_idx = list(range(len(sign_data)))
random.shuffle(_s_idx)
_s_split = int(0.9 * len(_s_idx))
sign_train = [sign_data[i] for i in _s_idx[:_s_split]]
sign_eval  = [sign_data[i] for i in _s_idx[_s_split:]]
print(f"  Sign Train: {len(sign_train)} | Sign Eval: {len(sign_eval)}")

Loaded 1595 English sign VQA samples.
  Sign Train: 1435 | Sign Eval: 160


In [10]:
# ════════════════════════════════════════════════════════════════════════════════
#  Mix video QA + traffic sign QA into a single English training set
#  Cache: mixed_train_en.json (input base or working dir)
# ════════════════════════════════════════════════════════════════════════════════
import json
import os
import random

_INPUT_BASE      = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_mixed_cache_in  = os.path.join(_INPUT_BASE, "mixed_train_en.json")
_mixed_cache_wk  = "/kaggle/working/mixed_train_en.json"

if os.path.exists(_mixed_cache_in):
    with open(_mixed_cache_in, "r", encoding="utf-8") as f:
        mixed_dataset = json.load(f)
    print(f"✅ Loaded mixed dataset from input ({len(mixed_dataset)} entries)")
elif os.path.exists(_mixed_cache_wk):
    with open(_mixed_cache_wk, "r", encoding="utf-8") as f:
        mixed_dataset = json.load(f)
    print(f"✅ Loaded mixed dataset from working ({len(mixed_dataset)} entries)")
else:
    # Mix video QA (converted_dataset) + traffic sign QA (sign_data)
    mixed_dataset = converted_dataset + sign_data
    random.seed(42)
    random.shuffle(mixed_dataset)
    with open(_mixed_cache_wk, "w", encoding="utf-8") as f:
        json.dump(mixed_dataset, f, ensure_ascii=False)
    print(f"✅ Mixed {len(converted_dataset)} video QA + {len(sign_data)} sign QA = {len(mixed_dataset)} total")
    print(f"   Saved to {_mixed_cache_wk}")

# 90/10 train/eval split
random.seed(42)
_m_idx   = list(range(len(mixed_dataset)))
random.shuffle(_m_idx)
_m_split = int(0.9 * len(_m_idx))
mixed_train = [mixed_dataset[i] for i in _m_idx[:_m_split]]
mixed_eval  = [mixed_dataset[i] for i in _m_idx[_m_split:]]
print(f"  Mixed Train: {len(mixed_train)} | Mixed Eval: {len(mixed_eval)}")


✅ Loaded mixed dataset from input (2759 entries)
  Mixed Train: 2483 | Mixed Eval: 276


In [11]:
if _skip_training:
    print("⏭ Skipping LoRA config + SFTTrainer setup (mixed model already cached).")
else:
    from unsloth.trainer import UnslothVisionDataCollator
    from trl import SFTTrainer, SFTConfig

    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = True,
        finetune_language_layers   = True,
        finetune_attention_modules = True,
        finetune_mlp_modules       = True,
        r = 16,
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )

    FastVisionModel.for_training(model)

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        data_collator = UnslothVisionDataCollator(model, tokenizer),
        train_dataset = mixed_train,
        eval_dataset  = mixed_eval,
        args = SFTConfig(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            num_train_epochs = 3,
            warmup_ratio = 0.05,
            learning_rate = 2e-4,
            logging_steps = 10,
            eval_strategy = "epoch",
            save_strategy = "epoch",
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "cosine",
            seed = 3407,
            output_dir = "outputs",
            report_to = "none",
            remove_unused_columns = False,
            dataset_text_field = "",
            dataset_kwargs = {"skip_prepare_dataset": True},
            max_length = 1024,
        ),
    )


⏭ Skipping LoRA config + SFTTrainer setup (mixed model already cached).


In [12]:
if _skip_training:
    print("⏭ Skipping trainer.train() (mixed model already cached).")
else:
    trainer_stats = trainer.train()


⏭ Skipping trainer.train() (mixed model already cached).


In [13]:
if _skip_training:
    print("⏭ Skipping save qwen2-vl-7b-mixed-en (already cached in input).")
else:
    print("Saving English mixed-trained model ...")
    model.save_pretrained("qwen2-vl-7b-mixed-en")
    tokenizer.save_pretrained("qwen2-vl-7b-mixed-en")


⏭ Skipping save qwen2-vl-7b-mixed-en (already cached in input).


# Inference preparation

In [14]:
model = FastVisionModel.for_inference(model)

In [15]:
# Load and translate test data to English
import json
import os

_INPUT_BASE           = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_cached_test_en       = os.path.join(_INPUT_BASE, "test_en.json")
_working_test_en      = "/kaggle/working/test_en.json"  # generated on previous run
_cached_test_prefixed = os.path.join(_INPUT_BASE, "test_prefixed.json")

if os.path.exists(_cached_test_en):
    test_prefixed_path = _cached_test_en
    print(f"✅ Using cached English test data from input: {test_prefixed_path}")
elif os.path.exists(_working_test_en):
    test_prefixed_path = _working_test_en
    print(f"✅ Using cached English test data from working: {test_prefixed_path}")
else:
    # Load original test data
    if os.path.exists(_cached_test_prefixed):
        src_path = _cached_test_prefixed
    else:
        prefix = "/kaggle/input/train3/train/"
        with open(test_split_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data["data"]:
            item["video_path"] = prefix + item["video_path"]
        src_path = "/kaggle/working/test_prefixed.json"
        with open(src_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    with open(src_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Translate questions, choices, answers
    print(f"Translating {len(data['data'])} test items to English...")
    all_q = [item["question"] for item in data["data"]]
    all_a = [item.get("answer", "") for item in data["data"]]
    all_c_flat = []
    c_map = []
    for i, item in enumerate(data["data"]):
        for j, c in enumerate(item.get("choices", [])):
            all_c_flat.append(c)
            c_map.append((i, j))

    trans_q = batch_translate(all_q)
    trans_a = batch_translate(all_a)
    trans_c = batch_translate(all_c_flat)
    save_translation_cache()

    # Apply translations (keep originals in _vi fields for reference)
    for idx, item in enumerate(data["data"]):
        item["question_vi"] = item["question"]
        item["question"] = trans_q[idx]
        item["answer_vi"] = item.get("answer", "")
        item["answer"] = trans_a[idx]

    trans_choices_per = [[] for _ in data["data"]]
    for flat_idx, (item_idx, _) in enumerate(c_map):
        trans_choices_per[item_idx].append(trans_c[flat_idx])
    for idx, item in enumerate(data["data"]):
        item["choices_vi"] = item.get("choices", [])
        item["choices"] = trans_choices_per[idx]

    test_prefixed_path = _working_test_en
    with open(test_prefixed_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"✅ Saved English test data → {test_prefixed_path}")


✅ Using cached English test data from input: /kaggle/input/datasets/huyqn12/cropped-zalo/test_en.json


In [16]:
with open(test_prefixed_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)["data"]
print(f"✅ Loaded {len(test_data)} English test items.")

✅ Loaded 326 English test items.


In [17]:
import os
import re
import cv2
import json
import torch
from unsloth import FastVisionModel

frames_save_dir = "/kaggle/working/extracted_frames"
os.makedirs(frames_save_dir, exist_ok=True)

model = FastVisionModel.for_inference(model)

# ── Answer label extractor ────────────────────────────────────────────────────
def extract_label(value):
    if not isinstance(value, str) or not value.strip():
        return None
    text = value.strip()
    if text[0] in "ABCD":
        return text[0]
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)
    m = re.search(r'\b([abcd])\b', text)
    if m:
        return m.group(1).upper()
    return None

# ── Shared instruction strings ────────────────────────────────────────────────
_DEFAULT_INSTRUCTION = (
    "You are an expert at answering multiple-choice questions about road traffic videos. "
    "You will first see key frames from a video, then cropped images of detected traffic signs (if any). "
    "Based on all the visual content, select the best answer (A, B, C, or D). "
    "Answer with ONLY the letter of the correct choice.\n\n"
)

_ITERATIVE_FIRST_PASS_INSTRUCTION = (
    "You are a traffic video analyst. "
    "Examine the key video frames and any detected traffic sign crops provided. "
    "If you are confident in your answer from visual context alone, respond with ONLY the "
    "single answer letter (A, B, C, or D). "
    "If the visual information is insufficient, respond with exactly: UNSURE\n\n"
)

# ── Inference function (instruction is now configurable) ─────────────────────
def inference_with_images(model, tokenizer, image_urls, prompt,
                          max_new_tokens=128, instruction=None):
    """Run VLM inference. Pass instruction=None to use the default."""
    if instruction is None:
        instruction = _DEFAULT_INSTRUCTION

    messages = [{
        "role": "user",
        "content": [
            *[{"type": "image", "image": url} for url in image_urls],
            {"type": "text", "text": instruction + prompt}
        ]
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=False,
        max_length=None,
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    answer_ids = generated_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

print("✅ Inference utilities ready (English, configurable instruction).")


✅ Inference utilities ready (English, configurable instruction).


# CLIP (base) + YOLO + Key Frame Selection

In [18]:
# Already installed in cell 1, just import
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel
print("✅ ultralytics + CLIP imports ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ ultralytics + CLIP imports ready.


In [19]:
# Use base CLIP directly — no fine-tuning (dataset too small to benefit)
import torch

_CLIP_BASE  = "openai/clip-vit-base-patch32"
_clip_ft_path = _CLIP_BASE
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Using base CLIP: {_clip_ft_path}")

✅ Using base CLIP: openai/clip-vit-base-patch32


In [20]:
# ════════════════════════════════════════════════════════════════════════════════
#  IMPROVED RAG — English corpus | BM25 + SBERT + CLIP visual + RRF
#  CLIP stays on GPU throughout inference (no CPU/GPU ping-pong).
# ════════════════════════════════════════════════════════════════════════════════
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPModel
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
from rank_bm25 import BM25Okapi

_MULTILINGUAL_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RAG_TOP_K = 5
_RRF_K    = 60
device    = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load multilingual encoder ─────────────────────────────────────────────────
print(f"Loading encoder: {_MULTILINGUAL_MODEL} ...")
_sbert_tok = AutoTokenizer.from_pretrained(_MULTILINGUAL_MODEL)
_sbert_enc = AutoModel.from_pretrained(_MULTILINGUAL_MODEL).to(device).eval()
print("✓ Encoder loaded.")

def _mean_pool(model_out, attention_mask):
    tok_emb = model_out.last_hidden_state
    mask    = attention_mask.unsqueeze(-1).expand(tok_emb.size()).float()
    return (tok_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def _sbert_encode(texts, batch_size=64):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = _sbert_tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = _sbert_enc(**enc)
        emb = F.normalize(_mean_pool(out, enc["attention_mask"]), p=2, dim=-1)
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0)

# ── Load reference dataset ────────────────────────────────────────────────────
print("Loading ghbihuy/vietnam_traffic_sign ...")
try:
    ref_ds = load_dataset("parquet", data_files="/kaggle/input/ref_traffic_sign_rules/**/*.parquet", split="train")
except Exception:
    ref_ds = load_dataset("ghbihuy/vietnam_traffic_sign", split="train")
print(f"  → {len(ref_ds)} rows | columns: {ref_ds.column_names}")

# ── Translate reference corpus to English (cached) ────────────────────────────
_INPUT_BASE = "/kaggle/input/datasets/huyqn12/cropped-zalo"
_REF_EN_CACHE = os.path.join(_INPUT_BASE, "ref_corpus_en.json")

if os.path.exists(_REF_EN_CACHE):
    import json
    with open(_REF_EN_CACHE, "r") as f:
        corpus_strings = json.load(f)
    print(f"✅ Loaded English reference corpus from cache ({len(corpus_strings)} entries)")
else:
    print("Translating reference corpus to English...")
    corpus_strings_vi = []
    for _row in ref_ds:
        _cat  = (_row.get("category")    or "").strip()
        _mean = (_row.get("meaning")     or "").strip()
        _desc = (_row.get("description") or "").strip()
        corpus_strings_vi.append(f"{_cat} | {_mean} | {_desc}")
    corpus_strings = batch_translate(corpus_strings_vi)
    save_translation_cache()
    # Save for reuse
    import json
    with open("/kaggle/working/ref_corpus_en.json", "w") as f:
        json.dump(corpus_strings, f, ensure_ascii=False)
    print(f"✅ Saved English corpus ({len(corpus_strings)} entries)")

# ── BM25 index (English) ──────────────────────────────────────────────────────
print("Building BM25 index (English) ...")
_bm25 = BM25Okapi([s.lower().split() for s in corpus_strings])

# ── Dense SBERT index ─────────────────────────────────────────────────────────
_SBERT_CACHE_INPUT   = os.path.join(_INPUT_BASE, "sbert_ref_embeds_en.pt")
_SBERT_CACHE_WORKING = "/kaggle/working/sbert_ref_embeds_en.pt"

if os.path.exists(_SBERT_CACHE_INPUT):
    _sbert_ref = torch.load(_SBERT_CACHE_INPUT, map_location="cpu")
    print(f"✅ SBERT embeddings loaded ({_sbert_ref.shape[0]} rows)")
elif os.path.exists(_SBERT_CACHE_WORKING):
    _sbert_ref = torch.load(_SBERT_CACHE_WORKING, map_location="cpu")
    print(f"✅ SBERT embeddings loaded ({_sbert_ref.shape[0]} rows)")
else:
    print("Encoding English corpus with SBERT ...")
    _sbert_ref = _sbert_encode(corpus_strings)
    torch.save(_sbert_ref, _SBERT_CACHE_WORKING)
    print(f"✅ Saved SBERT embeddings")

# ── CLIP visual index — stays on GPU ─────────────────────────────────────────
print(f"Loading fine-tuned CLIP from {_clip_ft_path} ...")
_clip_proc  = CLIPProcessor.from_pretrained(_clip_ft_path)
_clip_model = CLIPModel.from_pretrained(_clip_ft_path).to(device).eval()

def _clip_get_image_feats(pixel_values):
    """Safely extract image features tensor from CLIP output."""
    out = _clip_model.get_image_features(pixel_values=pixel_values)
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("image_embeds", "pooler_output", "last_hidden_state"):
        if hasattr(out, attr):
            t = getattr(out, attr)
            if isinstance(t, torch.Tensor):
                return t[:, 0] if t.dim() == 3 else t
    t = out[0]
    return t[:, 0] if (isinstance(t, torch.Tensor) and t.dim() == 3) else t

def _clip_get_text_feats(**kwargs):
    """Safely extract text features tensor from CLIP output."""
    out = _clip_model.get_text_features(**kwargs)
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("text_embeds", "pooler_output", "last_hidden_state"):
        if hasattr(out, attr):
            t = getattr(out, attr)
            if isinstance(t, torch.Tensor):
                return t[:, 0] if t.dim() == 3 else t
    t = out[0]
    return t[:, 0] if (isinstance(t, torch.Tensor) and t.dim() == 3) else t

_CLIP_CACHE_INPUT   = os.path.join(_INPUT_BASE, "clip_img_embeds_en.pt")
_CLIP_CACHE_WORKING = "/kaggle/working/clip_img_embeds_en.pt"

if os.path.exists(_CLIP_CACHE_INPUT):
    _clip_img_embeds = torch.load(_CLIP_CACHE_INPUT, map_location="cpu")
    print(f"✅ CLIP image embeddings loaded ({_clip_img_embeds.shape[0]} rows)")
elif os.path.exists(_CLIP_CACHE_WORKING):
    _clip_img_embeds = torch.load(_CLIP_CACHE_WORKING, map_location="cpu")
    print(f"✅ CLIP image embeddings loaded ({_clip_img_embeds.shape[0]} rows)")
else:
    print("Encoding reference images with CLIP ...")
    _clip_img_embeds = []
    with torch.no_grad():
        for _i in tqdm(range(0, len(ref_ds), 32), desc="CLIP ref images"):
            _imgs = [img.convert("RGB") for img in ref_ds[_i:_i+32]["image"]]
            _inp  = _clip_proc(images=_imgs, return_tensors="pt").to(device)
            _f    = F.normalize(_clip_get_image_feats(_inp.pixel_values), p=2, dim=-1)
            _clip_img_embeds.append(_f.cpu())
    _clip_img_embeds = torch.cat(_clip_img_embeds, dim=0)
    torch.save(_clip_img_embeds, _CLIP_CACHE_WORKING)
    print(f"✅ Saved CLIP embeddings")

# NOTE: CLIP stays on GPU — no .cpu() call here!

# ── Rank helpers ──────────────────────────────────────────────────────────────
def _bm25_rank(query, top_n=15):
    scores = _bm25.get_scores(query.lower().split())
    return sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]

def _sbert_rank(query, top_n=15):
    q    = _sbert_encode([query]).to(device)
    sims = (q @ _sbert_ref.to(device).T).squeeze(0)
    return torch.topk(sims, min(top_n, sims.shape[0])).indices.tolist()

def _clip_img_rank(pil_crops, top_n=15):
    """CLIP visual ranking — model already on GPU."""
    with torch.no_grad():
        _inp  = _clip_proc(images=pil_crops, return_tensors="pt").to(device)
        feats = F.normalize(_clip_get_image_feats(_inp.pixel_values), p=2, dim=-1).cpu()
    sims = torch.matmul(feats, _clip_img_embeds.T)    # (C, N)
    return torch.topk(sims.max(dim=0).values, min(top_n, sims.shape[1])).indices.tolist()

def _clip_text_rank(query, top_n=15):
    """CLIP text → image ranking for cross-modal retrieval."""
    with torch.no_grad():
        _inp = _clip_proc(text=[query], return_tensors="pt", padding=True, truncation=True).to(device)
        feats = F.normalize(_clip_get_text_feats(**_inp), p=2, dim=-1).cpu()
    sims = (feats @ _clip_img_embeds.T).squeeze(0)
    return torch.topk(sims, min(top_n, sims.shape[0])).indices.tolist()

def _rrf(rankings, k=_RRF_K):
    scores = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=lambda i: scores[i], reverse=True)

def _fmt(idx):
    return corpus_strings[idx]

# ── Public RAG function ───────────────────────────────────────────────────────
def retrieve_rag_context(
    pil_crops: list,
    question: str,
    choices: list,
    top_k: int = RAG_TOP_K,
    detected_classes: list = None,
) -> list:
    """
    Multi-signal RAG with RRF fusion (English).
    Signals: BM25 + SBERT dense + CLIP visual + CLIP text-to-image.
    """
    N = max(top_k * 4, 15)
    rankings = []
    
    # Nếu YOLO phát hiện được biển báo, dùng chính tên biển báo để search (chính xác nhất)
    if detected_classes:
        for sign_class in detected_classes:
            if sign_class:
                rankings.append(_bm25_rank(sign_class, top_n=N))
                rankings.append(_sbert_rank(sign_class, top_n=N))
                rankings.append(_clip_text_rank(sign_class, top_n=N))
    else:
        # Nếu YOLO không phát hiện được gì, DÙNG CÂU HỎI thay vì list(choices)
        # Việc dùng list(choices) gây nhiễu vì đưa vào các đáp án sai.
        rankings.append(_bm25_rank(question, top_n=N))
        rankings.append(_sbert_rank(question, top_n=N))
        rankings.append(_clip_text_rank(question, top_n=N))
    
    # CLIP visual from crops
    if pil_crops:
        rankings.append(_clip_img_rank(pil_crops[:8], top_n=N))

    if not rankings:
        return []

    fused = _rrf(rankings)[:top_k]
    return [_fmt(i) for i in fused]

# ── CLIP-based frame scoring for key frame selection ──────────────────────────
def score_frame_with_clip(frame_pil, question_text):
    """Score a single frame's relevance to the question using CLIP."""
    with torch.no_grad():
        img_inp = _clip_proc(images=[frame_pil], return_tensors="pt").to(device)
        txt_inp = _clip_proc(text=[question_text], return_tensors="pt", padding=True, truncation=True).to(device)
        img_feat = F.normalize(_clip_get_image_feats(img_inp.pixel_values), p=2, dim=-1)
        txt_feat = F.normalize(_clip_get_text_feats(**txt_inp), p=2, dim=-1)
    return (img_feat @ txt_feat.T).item()

print("✅ English RAG ready (BM25 + SBERT + CLIP visual + CLIP text + RRF).")

Loading encoder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CLIP image embeddings loaded (250 rows)
✅ English RAG ready (BM25 + SBERT + CLIP visual + CLIP text + RRF).


In [21]:
# ════════════════════════════════════════════════════════════════════════════════
#  VERSION 7 MODIFIED: STRICT MICRO-HINT RAG & PROMPT-ALIGNED BASELINE
#  - Pipeline C: Prompt updated to match Pipeline B's high-performing style 
#                (but strictly No-RAG, removing the 'UNSURE' fallback).
#  - Pipeline A: 100% Untouched (Micro-Hint RAG).
#  - Pipeline B: 100% Untouched (Logit-Gated with threshold raised to 0.85).
# ════════════════════════════════════════════════════════════════════════════════
import os
import cv2
import csv
import json
import torch
import re
import numpy as np
import torch.nn.functional as F
from collections import defaultdict
from ultralytics import YOLO
from tqdm import tqdm
from PIL import Image

# ── Init YOLO + Cache ─────────────────────────────────────────────────────────
yolo_model  = YOLO("/kaggle/input/besttraffic-real/best (2).pt")
class_names = yolo_model.names
_class_names_en = {k: translate_vi_to_en(str(v)) for k, v in class_names.items()}
save_translation_cache()

crop_save_dir      = "/kaggle/working/cropped_signs"
fullframe_save_dir = "/kaggle/working/full_frames"
frames_log_path    = "/kaggle/working/extracted_frames_log.jsonl"
os.makedirs(crop_save_dir, exist_ok=True)
os.makedirs(fullframe_save_dir, exist_ok=True)
open(frames_log_path, "w").close()

MAX_HISTORY = 3
_VLM_MAX_SIZE = 1280

def sanitize_filename(s):
    return re.sub(r'[./\\\\:*?\"<>|]', '_', s)

# ── HÀM RAG SIÊU KHẮT KHE: MICRO-OCR HINT ─────────────────────────────────────
def get_micro_ocr_hint(detected_classes):
    """
    Chỉ bắt các biển báo thực sự là rào cản thị giác của VLM: Tốc độ, Tải trọng, Chiều cao.
    Các con số khác (như mã số biển báo) bị loại bỏ hoàn toàn.
    """
    if not detected_classes: 
        return ""
    
    # Từ khóa chỉ nhắm vào đại lượng đo lường thực tế
    strict_keywords = ["km/h", "speed limit", "tấn", "ton", "meter", "m", "chiều cao", "tải trọng"]
    
    for sign in detected_classes:
        s_lower = sign.lower()
        if any(k in s_lower for k in strict_keywords):
            # Lấy đúng 1 biển báo rõ nhất và trả về định dạng "thì thầm" nội tuyến
            return f" (Zoom Context: {sign})"
            
    return ""

# ── Hàm Inference trích xuất Logit Confidence ────────────────────────────────
def inference_with_logits_gating(model, tokenizer, image_urls, prompt, instruction=None):
    if instruction is None:
        instruction = _DEFAULT_INSTRUCTION

    messages = [{
        "role": "user",
        "content": [
            *[{"type": "image", "image": url} for url in image_urls],
            {"type": "text", "text": instruction + prompt}
        ]
    }]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            temperature=0.0,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs.sequences[0, inputs["input_ids"].shape[1]:]
    decoded_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    confidence = 1.0
    if hasattr(outputs, "scores") and len(outputs.scores) > 0:
        first_token_logits = outputs.scores[0][0]
        probs = F.softmax(first_token_logits, dim=-1)
        confidence = float(probs[gen_tokens[0]].item())

    return decoded_text, confidence

# ── Key Frame Selection logic ─────────────────────────────────────────────────
def select_key_frames(video_path, question_text, video_id, num_candidates=20, max_key_frames=5, max_size=_VLM_MAX_SIZE):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return [], [], {}
    candidate_indices = np.linspace(0, total_frames - 1, num_candidates, dtype=int).tolist()
    candidates = []
    for idx in candidate_indices:
        if not cap.set(cv2.CAP_PROP_POS_FRAMES, idx): continue
        ret, frame = cap.read()
        if not ret or frame is None or frame.size == 0: continue
        h, w = frame.shape[:2]
        scale = min(1.0, max_size / max(h, w))
        frame_resized = cv2.resize(frame, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
        candidates.append((idx, frame_resized, Image.fromarray(cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB))))
    cap.release()
    if not candidates: return [], [], {}

    clip_scores, yolo_results_per_frame = [], []
    for frame_idx, frame_bgr, frame_pil in candidates:
        clip_scores.append(score_frame_with_clip(frame_pil, question_text))
        # Giữ YOLO ngưỡng an toàn 0.55
        res = yolo_model(frame_bgr, conf=0.55, iou=0.45, verbose=False)
        yolo_results_per_frame.append(res)

    sorted_idx = np.argsort(clip_scores)[::-1]
    selected, selected_fi = [], []
    for si in sorted_idx:
        if len(selected) >= max_key_frames: break
        fi = candidates[si][0]
        if any(abs(fi - prev) < max(total_frames // 8, 5) for prev in selected_fi): continue
        selected.append(si)
        selected_fi.append(fi)
    for si in sorted_idx:
        if si not in selected and len(selected) < max_key_frames: selected.append(si)
    selected.sort(key=lambda si: candidates[si][0])

    key_frame_paths, all_crops_info, unique_crops = [], [], {}
    seen_classes = set()
    for si in selected:
        frame_idx, frame_bgr, _ = candidates[si]
        fp = os.path.join(fullframe_save_dir, f"{video_id}_key_{frame_idx}.jpg")
        cv2.imwrite(fp, frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        key_frame_paths.append(fp)

        boxes = yolo_results_per_frame[si][0].boxes
        if boxes is None or len(boxes) == 0: continue
        h_f, w_f = frame_bgr.shape[:2]
        for box in boxes:
            cls_id = int(box.cls[0].item())
            cn_en  = _class_names_en.get(cls_id, f"class_{cls_id}")
            if cn_en in seen_classes and sum(1 for _, c in all_crops_info if c == cn_en) >= 2: continue
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            margin = 30
            crop = frame_bgr[max(0, y1-margin):min(h_f, y2+margin), max(0, x1-margin):min(w_f, x2+margin)]
            if crop.size == 0: continue
            
            safe_name = re.sub(r'[./\\\\:*?\"<>|]', '_', cn_en)
            cp = os.path.join(crop_save_dir, f"{video_id}_{frame_idx}_{safe_name}.jpg")
            cv2.imwrite(cp, crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            all_crops_info.append((cp, cn_en))
            seen_classes.add(cn_en)
            if cn_en not in unique_crops: unique_crops[cn_en] = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    return key_frame_paths, all_crops_info, unique_crops

# ════════════════════════════════════════════════════════════════════════════════
#  VÒNG LẶP CHẠY INFERENCE CHÍNH
# ════════════════════════════════════════════════════════════════════════════════
results_A, results_B, results_C = [], [], []
history_dict = defaultdict(list)
correct_A = correct_B = correct_C = total = rag_triggered = 0

for item in tqdm(test_data, desc="Evaluating Micro-Hint RAG"):
    video_path = item["video_path"]
    video_id   = os.path.splitext(os.path.basename(video_path))[0]
    question   = item["question"]
    choices    = item.get("choices", [])
    gt_answer  = extract_label(item.get("answer", ""))

    key_frame_paths, crops_info, unique_crops = select_key_frames(video_path, question, video_id)
    if not key_frame_paths: continue

    detected_classes = list(set([cn for _, cn in crops_info]))
    all_vlm_images   = key_frame_paths + [cp for cp, _ in crops_info]

    history_prompt = ""
    vh = history_dict.get(video_path, [])[-MAX_HISTORY:]
    if vh: history_prompt = "Previous Q&A from this video:\n" + "\n".join(vh) + "\n\n"

    try:
        # ─── PIPELINE C: PURE NO-RAG BASELINE (CẬP NHẬT THEO STYLE CỦA A & B) ─
        # Sử dụng đúng instruction và format mạnh nhất, nhưng loại bỏ yếu tố RAG
        prompt_C = (
            f"{history_prompt}"
            f"Question: {question}\n"
            f"Choices:\n" + "\n".join(choices) + "\n"
            f"\nAnswer ONLY with the letter (A/B/C/D)."
        )
        resp_C, _ = inference_with_logits_gating(model, tokenizer, all_vlm_images, prompt_C, instruction=_ITERATIVE_FIRST_PASS_INSTRUCTION)
        pred_C    = extract_label(resp_C) or "A"

        # ─── PIPELINE A: MICRO-HINT RAG (GIỮ NGUYÊN 100%) ─────────────────────
        micro_ocr_hint = get_micro_ocr_hint(detected_classes)
        prompt_A = (
            f"{history_prompt}"
            f"Question: {question}{micro_ocr_hint}\n"   # Gắn gợi ý nhẹ nhàng ngay sau câu hỏi
            f"Choices:\n" + "\n".join(choices) + "\n"
            f"\nSelect the correct answer (A, B, C, or D)."
        )
        resp_A, _ = inference_with_logits_gating(model, tokenizer, all_vlm_images, prompt_A, instruction=None)
        pred_A    = extract_label(resp_A) or "A"

        # ─── PIPELINE B: GATED ITERATIVE RAG (GIỮ NGUYÊN 100%) ────────────────
        prompt_B1 = (
            f"{history_prompt}"
            f"Question: {question}\n"
            f"Choices:\n" + "\n".join(choices) + "\n"
            f"\nAnswer ONLY with the letter (A/B/C/D) if certain, or 'UNSURE' if not."
        )
        resp_B1, confidence = inference_with_logits_gating(model, tokenizer, all_vlm_images, prompt_B1, instruction=_ITERATIVE_FIRST_PASS_INSTRUCTION)
        pred_B = extract_label(resp_B1) or "A"
        
        # CHỈ KÍCH HOẠT LƯỢT 2 NẾU: Tự tin < 0.85 VÀ có Micro-Hint về số liệu
        if confidence < 0.85 and micro_ocr_hint:
            rag_triggered += 1
            # Pass 2 gọi lại cấu trúc siêu gọn của Pipeline A
            prompt_B2 = prompt_A 
            resp_B2, _ = inference_with_logits_gating(model, tokenizer, all_vlm_images, prompt_B2, instruction=None)
            pred_B     = extract_label(resp_B2) or "A"

        # ─── Kiểm tra kết quả ─────────────────────────────────────────────────
        if pred_C == gt_answer: correct_C += 1
        if pred_A == gt_answer: correct_A += 1
        if pred_B == gt_answer: correct_B += 1
        total += 1

        results_A.append({"id": item["id"], "answer": pred_A})
        results_B.append({"id": item["id"], "answer": pred_B})
        results_C.append({"id": item["id"], "answer": pred_C})
        
        # History trung lập lấy từ Pipeline C
        history_dict[video_path].append(f"Q: {question} → A: {resp_C.strip()[:80]}")

        if total % 10 == 0:
            print(
                f"[{total}] C(Base): {correct_C/total:.4f} | "
                f"A(Micro-RAG): {correct_A/total:.4f} | "
                f"B(Gated Micro): {correct_B/total:.4f} "
                f"[Triggers: {rag_triggered}]"
            )

    except Exception as e:
        print(f"[Error] {video_id}: {e}")
        continue
    finally:
        torch.cuda.empty_cache()

# ─── Xuất file nộp bài ────────────────────────────────────────────────────────
print("\n=== KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG ===")
print(f"Pipeline C (NO RAG Baseline)     : {correct_C/total:.4f} ({correct_C}/{total})")
print(f"Pipeline A (Micro-Hint RAG)      : {correct_A/total:.4f} ({correct_A}/{total})")
print(f"Pipeline B (Gated Micro RAG)     : {correct_B/total:.4f} ({correct_B}/{total}) [Triggers: {rag_triggered}]")

best_results = results_A if correct_A >= max(correct_C, correct_B) else (results_B if correct_B >= correct_C else results_C)
winner = "Pipeline A" if best_results == results_A else ("Pipeline B" if best_results == results_B else "Pipeline C")

with open("/kaggle/working/submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "answer"])
    writer.writeheader()
    writer.writerows(best_results)
print(f"✓ Đã chọn kết quả từ {winner} để lưu vào submission.csv.")

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

💾 Saved translation cache (6275 entries)


Evaluating Micro-Hint RAG:   3%|▎         | 10/326 [07:55<3:39:12, 41.62s/it]

[10] C(Base): 0.7000 | A(Micro-RAG): 0.8000 | B(Gated Micro): 0.8000 [Triggers: 2]


Evaluating Micro-Hint RAG:   6%|▌         | 20/326 [16:07<3:58:53, 46.84s/it]

...

[310] C(Base): 0.7387 | A(Micro-RAG): 0.7516 | B(Gated Micro): 0.7484 [Triggers: 33]


Evaluating Micro-Hint RAG:  98%|█████████▊| 320/326 [4:14:14<05:40, 56.77s/it]

[320] C(Base): 0.7344 | A(Micro-RAG): 0.7438 | B(Gated Micro): 0.7438 [Triggers: 33]


Evaluating Micro-Hint RAG: 100%|██████████| 326/326 [4:18:19<00:00, 47.54s/it]


=== KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG ===
Pipeline C (NO RAG Baseline)     : 0.7393 (241/326)
Pipeline A (Micro-Hint RAG)      : 0.7485 (244/326)
Pipeline B (Gated Micro RAG)     : 0.7485 (244/326) [Triggers: 33]
✓ Đã chọn kết quả từ Pipeline A để lưu vào submission.csv.


## Pipeline Comparison Test (single sample)
- **Pipeline A — No RAG**: key frames only (CLIP+YOLO selected)
- **Pipeline B — Full RAG**: BM25 + SBERT + CLIP visual + CLIP text + RRF
- All prompts in English, no sign class names injected

In [22]:
# ════════════════════════════════════════════════════════════════════════════════
#  Pipeline Comparison — single test item
# ════════════════════════════════════════════════════════════════════════════════
import random as _random
from PIL import Image

TEST_ITEM_ID = None  # Set to a specific ID string, or None for random

if TEST_ITEM_ID is not None:
    test_item = next((x for x in test_data if str(x["id"]) == str(TEST_ITEM_ID)), None)
    if test_item is None:
        raise ValueError(f"TEST_ITEM_ID {TEST_ITEM_ID!r} not found")
else:
    test_item = _random.choice(test_data)

print("=" * 72)
print(f"ITEM ID  : {test_item['id']}")
print(f"VIDEO    : {test_item.get('video_path', 'N/A')}")
print(f"QUESTION : {test_item['question']}")
print("CHOICES  :")
for c in test_item.get("choices", []):
    print(f"  {c}")
print(f"TRUE ANS : {test_item.get('answer', 'N/A')}")
print("=" * 72)

# ── Shared: CLIP+YOLO key frame selection ─────────────────────────────────────
_vid = os.path.splitext(os.path.basename(test_item["video_path"]))[0]
print("\n[Shared] Selecting key frames with CLIP+YOLO ...")
_kf_paths, _crops, _ucrop = select_key_frames(
    test_item["video_path"], test_item["question"], _vid + "_cmp",
    num_candidates=20, max_key_frames=4
)
print(f"  Key frames: {len(_kf_paths)}")
print(f"  Unique signs detected: {list(_ucrop.keys())}")

# ══════════════════════════════════════════════════════════════════════════════
#  Pipeline A — No RAG (just key frames + question)
# ══════════════════════════════════════════════════════════════════════════════
def run_pipeline_A(item, kf_paths):
    prompt = (
        f"Question: {item['question']}\n"
        f"Choices:\n" + "\n".join(item.get("choices", [])) + "\n"
        f"\nSelect the correct answer (A, B, C, or D)."
    )
    resp = inference_with_images(model, tokenizer, kf_paths, prompt, max_new_tokens=128)
    return extract_label(resp.strip()), resp.strip()

# ══════════════════════════════════════════════════════════════════════════════
#  Pipeline B — Full RAG (BM25 + SBERT + CLIP + RRF)
# ══════════════════════════════════════════════════════════════════════════════
def run_pipeline_B(item, kf_paths, unique_crops):
    pil_crops = [Image.fromarray(a) for a in unique_crops.values()][:8]
    rules = retrieve_rag_context(pil_crops, item["question"], item.get("choices", []), top_k=RAG_TOP_K)
    rag_ctx = ""
    if rules:
        rag_ctx = "Reference traffic regulations:\n- " + "\n- ".join(rules) + "\n"
    prompt = (
        f"{rag_ctx}"
        f"Question: {item['question']}\n"
        f"Choices:\n" + "\n".join(item.get("choices", [])) + "\n"
        f"\nSelect the correct answer (A, B, C, or D)."
    )
    resp = inference_with_images(model, tokenizer, kf_paths, prompt, max_new_tokens=128)
    return extract_label(resp.strip()), resp.strip(), rules

# ══════════════════════════════════════════════════════════════════════════════
#  RUN BOTH PIPELINES
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("[Pipeline A] No RAG — key frames + question only")
pred_A, resp_A = run_pipeline_A(test_item, _kf_paths)
print(f"  Response: {resp_A[:300]}")
print(f"  ▶ Predicted: {pred_A}")

print("\n" + "─" * 72)
print("[Pipeline B] Full English RAG (BM25 + SBERT + CLIP + RRF)")
pred_B, resp_B, rules_B = run_pipeline_B(test_item, _kf_paths, _ucrop)
print(f"  Retrieved rules:")
for i, r in enumerate(rules_B, 1):
    print(f"    [{i}] {r[:120]}")
print(f"  Response: {resp_B[:300]}")
print(f"  ▶ Predicted: {pred_B}")

# ══════════════════════════════════════════════════════════════════════════════
#  COMPARISON
# ══════════════════════════════════════════════════════════════════════════════
true_ans = extract_label(test_item.get("answer", ""))
print("\n" + "=" * 72)
print("COMPARISON SUMMARY")
print("=" * 72)
print(f"  True Answer          : {true_ans}  ({test_item.get('answer','N/A')})")
print(f"  Pipeline A (No RAG)  : {pred_A}  {'✓' if pred_A == true_ans else '✗'}")
print(f"  Pipeline B (Full RAG): {pred_B}  {'✓' if pred_B == true_ans else '✗'}")
print("=" * 72)

# Cleanup
for fp in _kf_paths:
    if os.path.exists(fp): os.remove(fp)
for cp, _ in _crops:
    if os.path.exists(cp): os.remove(cp)

ITEM ID  : train_1314
VIDEO    : /kaggle/input/train3/train/train/videos/faf14ff0_094_clip_010_0066_0075_Y.mp4
QUESTION : In this video, what does the sign that appears mean?
CHOICES  :
  A. Signals that the upcoming road is flat and easy to go
  B. Signaling that you are approaching a road with many dangerous turns in a row
  C. Signaling an impending intersection with a priority road
  D. Signaling slippery roads when it rains
TRUE ANS : B. Signaling that you are approaching a road with many dangerous turns in a row

[Shared] Selecting key frames with CLIP+YOLO ...


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  Key frames: 4
  Unique signs detected: []

────────────────────────────────────────────────────────────────────────
[Pipeline A] No RAG — key frames + question only


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  Response: B
  ▶ Predicted: B

────────────────────────────────────────────────────────────────────────
[Pipeline B] Full English RAG (BM25 + SBERT + CLIP + RRF)
  Retrieved rules:
    [1] Additional signs | License plate S.503c "Direction of effect of the sign" | Meaning of sign S.503c (Direction of effect 
    [2] Directional signs | License plate I.409 "Turning place" | Meaning of sign I.409 "Turning place" 🚗↩️
- This is a sign (bl
    [3] Additional signs | License plate S.503d "Direction of effect of the sign" | Meaning of sign S.503d ⬇️
- This is the auxi
    [4] Directional signs | License plate I.407a "One-way street"
One-way street ahead. | Meaning of sign I.407a (One-way street
    [5] Additional signs | License plate S.503f "Direction of effect of the sign" | Meaning of sign S.503f
- This is the auxilia
  Response: B
  ▶ Predicted: B

COMPARISON SUMMARY
  True Answer          : B  (B. Signaling that you are approaching a road with many dangerous turns in a row)
  Pipeline A